# Parkinson's Disease Risk Prediction — New Model (17 Features)
### Real voice data from parkinsons.data + Clinically realistic questionnaire features
### Output: 0 = Healthy | 1 = Parkinson's Disease

## Step 1: Import Libraries

In [3]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score
)

np.random.seed(42)
print('Libraries loaded!')

ModuleNotFoundError: No module named 'matplotlib'

## Step 2: Load Real Dataset

In [ ]:
raw_df = pd.read_csv('parkinsons.data')
print('Shape:', raw_df.shape)
print('Class distribution:')
print(raw_df['status'].value_counts())
raw_df.head()

## Step 3: Extract the 5 Voice Features Used by Chatbot

| Chatbot Feature | Dataset Column     | Description                    |
|-----------------|--------------------|---------------------------------|
| `fo`            | `MDVP:Fo(Hz)`      | Average vocal fundamental freq |
| `jitter`        | `MDVP:Jitter(%)`   | Frequency variation            |
| `shimmer`       | `MDVP:Shimmer`     | Amplitude variation            |
| `nhr`           | `NHR`              | Noise-to-Harmonics Ratio       |
| `hnr`           | `HNR`              | Harmonics-to-Noise Ratio       |

In [ ]:
voice_df = raw_df[['MDVP:Fo(Hz)', 'MDVP:Jitter(%)', 'MDVP:Shimmer', 'NHR', 'HNR', 'status']].copy()
voice_df.rename(columns={
    'MDVP:Fo(Hz)'   : 'fo',
    'MDVP:Jitter(%)': 'jitter',
    'MDVP:Shimmer'  : 'shimmer',
    'NHR'           : 'nhr',
    'HNR'           : 'hnr'
}, inplace=True)

print('Voice feature stats by class:')
print(voice_df.groupby('status').mean().round(5))
voice_df.head()

## Step 4: Generate Questionnaire Features

The parkinsons.data file has no questionnaire columns.
We generate them using clinically realistic distributions per class.
Parkinson patients score higher (1-3), healthy subjects score low (0-1).

**Replace with real questionnaire data when collected.**

In [ ]:
SYMPTOM_COLS = [
    'resting_tremor', 'bradykinesia', 'muscle_stiffness', 'balance_problem',
    'walking_difficulty', 'handwriting_change', 'voice_change',
    'facial_expression_change', 'daily_activity_difficulty',
    'sleep_disorder', 'smell_loss', 'rigidity_issue'
]
VOICE_COLS = ['jitter', 'shimmer', 'fo', 'nhr', 'hnr']

# Mean scores based on MDS-UPDRS Part II clinical research
PK_MEANS = [2.2, 2.1, 2.0, 1.9, 2.0, 2.1, 1.9, 1.8, 2.0, 1.9, 2.2, 2.1]
HL_MEANS = [0.3, 0.3, 0.4, 0.3, 0.3, 0.4, 0.3, 0.3, 0.3, 0.4, 0.3, 0.3]

def add_questionnaire_features(df):
    result = df.copy()
    for col, pk_m, hl_m in zip(SYMPTOM_COLS, PK_MEANS, HL_MEANS):
        scores = np.where(
            result['status'] == 1,
            np.random.normal(pk_m, 0.85, len(result)),
            np.random.normal(hl_m, 0.40, len(result))
        )
        result[col] = np.clip(np.round(scores), 0, 3).astype(int)
    return result

full_df = add_questionnaire_features(voice_df)
print('Final dataset shape:', full_df.shape)
print('\nQuestionnaire means by class:')
print(full_df.groupby('status')[SYMPTOM_COLS].mean().round(2))
full_df.head()

## Step 5: Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = full_df['status'].value_counts()
axes[0].pie(counts, labels=["Parkinson's (1)", 'Healthy (0)'],
            autopct='%1.1f%%', colors=['#E74C3C','#2ECC71'], startangle=90)
axes[0].set_title('Class Distribution', fontsize=13)

m_pk = full_df[full_df['status']==1][SYMPTOM_COLS].mean()
m_hl = full_df[full_df['status']==0][SYMPTOM_COLS].mean()
x = np.arange(len(SYMPTOM_COLS))
axes[1].bar(x-0.2, m_pk, 0.4, label="Parkinson's", color='#E74C3C')
axes[1].bar(x+0.2, m_hl, 0.4, label='Healthy',     color='#2ECC71')
axes[1].set_xticks(x)
axes[1].set_xticklabels([c.replace('_',' ') for c in SYMPTOM_COLS],
                         rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('Mean Score (0-3)')
axes[1].set_title('Questionnaire Scores by Class', fontsize=13)
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Voice feature distributions (real data from parkinsons.data)
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, col in zip(axes, VOICE_COLS):
    for label, color, name in [(0,'#2ECC71','Healthy'), (1,'#E74C3C',"Parkinson's")]:
        ax.hist(full_df[full_df['status']==label][col], bins=20, alpha=0.6,
                color=color, label=name)
    ax.set_title(col, fontsize=11)
    ax.legend(fontsize=7)
plt.suptitle('Voice Feature Distributions by Class (Real Data)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
FEATURE_COLS = SYMPTOM_COLS + VOICE_COLS
plt.figure(figsize=(13, 10))
corr = full_df[FEATURE_COLS + ['status']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5, annot_kws={'size':7})
plt.title('Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.show()

## Step 6: Prepare Features and Labels

In [ ]:
# All 17 features in the exact order the chatbot sends them
FEATURE_COLS = [
    'resting_tremor', 'bradykinesia', 'muscle_stiffness', 'balance_problem',
    'walking_difficulty', 'handwriting_change', 'voice_change',
    'facial_expression_change', 'daily_activity_difficulty',
    'sleep_disorder', 'smell_loss', 'rigidity_issue',
    'jitter', 'shimmer', 'fo', 'nhr', 'hnr'
]

X = full_df[FEATURE_COLS]
y = full_df['status']

print('Feature matrix shape:', X.shape)
print('Missing values:', X.isnull().sum().sum())

## Step 7: Train/Test Split and Feature Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples')

## Step 8: Train and Compare Multiple Models

In [ ]:
models = {
    'SVM (Linear)'      : SVC(kernel='linear', probability=True, random_state=42),
    'SVM (RBF)'         : SVC(kernel='rbf',    probability=True, random_state=42),
    'Random Forest'     : RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting' : GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Logistic Reg.'     : LogisticRegression(max_iter=1000, random_state=42)
}

results = []
for name, clf in models.items():
    clf.fit(X_train_sc, y_train)
    tr_acc = accuracy_score(y_train, clf.predict(X_train_sc))
    te_acc = accuracy_score(y_test,  clf.predict(X_test_sc))
    cv     = cross_val_score(clf, X_train_sc, y_train, cv=5, scoring='accuracy')
    auc    = roc_auc_score(y_test, clf.predict_proba(X_test_sc)[:,1])
    results.append({'Model':name, 'Train Acc':round(tr_acc,4), 'Test Acc':round(te_acc,4),
                    'CV Mean':round(cv.mean(),4), 'CV Std':round(cv.std(),4), 'ROC-AUC':round(auc,4)})
    print(f'{name:22s} | Train:{tr_acc:.4f} | Test:{te_acc:.4f} | CV:{cv.mean():.4f}+/-{cv.std():.4f} | AUC:{auc:.4f}')

res_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
print('\n--- Sorted by ROC-AUC ---')
res_df

## Step 9: Best Model Detailed Evaluation

In [ ]:
best_name  = res_df.iloc[0]['Model']
best_model = models[best_name]
print(f'Best model selected: {best_name}')

y_pred = best_model.predict(X_test_sc)
y_prob = best_model.predict_proba(X_test_sc)[:,1]

print(f'Test Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC-AUC       : {roc_auc_score(y_test, y_prob):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Healthy (0)', "Parkinson's (1)"]))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Healthy', "Parkinson's"],
            yticklabels=['Healthy', "Parkinson's"])
plt.title(f'Confusion Matrix -- {best_name}', fontsize=13)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## Step 10: Feature Importance

In [ ]:
rf  = models['Random Forest']
imp = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

colors = ['#E74C3C' if f in VOICE_COLS else '#3498DB' for f in imp.index]
plt.figure(figsize=(9,7))
imp.plot(kind='barh', color=colors)
plt.title('Feature Importance (Random Forest)\nBlue = Questionnaire  |  Red = Voice', fontsize=12)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Top 5 most important features:')
print(imp.sort_values(ascending=False).head())

## Step 11: Save Model Files

In [ ]:
joblib.dump(best_model,   'parkinsons_model.pkl')
joblib.dump(scaler,       'parkinsons_scaler.pkl')
joblib.dump(FEATURE_COLS, 'feature_cols.pkl')

print('Saved:')
print('  parkinsons_model.pkl   -- trained ML model')
print('  parkinsons_scaler.pkl  -- StandardScaler fitted on training data')
print('  feature_cols.pkl       -- 17 feature names in correct order')
print(f'\nModel: {best_name}  |  Features: {len(FEATURE_COLS)}')

## Step 12: Chatbot Prediction Function

In [ ]:
def predict_risk(answers: dict) -> dict:
    """
    Predict Parkinson's risk from chatbot input.

    Parameters
    ----------
    answers : dict
        Questionnaire keys -> int 0/1/2/3
        Voice feature keys -> float from audio library

    Returns
    -------
    dict: ml_prediction, ml_confidence, symptom_score, risk_level, recommendation
    """
    model_  = joblib.load('parkinsons_model.pkl')
    scaler_ = joblib.load('parkinsons_scaler.pkl')
    cols_   = joblib.load('feature_cols.pkl')

    vec    = np.array([answers[c] for c in cols_]).reshape(1, -1)
    vec_sc = scaler_.transform(vec)

    pred       = model_.predict(vec_sc)[0]
    confidence = model_.predict_proba(vec_sc)[0][1]   # P(Parkinson's)

    q_cols = [
        'resting_tremor','bradykinesia','muscle_stiffness','balance_problem',
        'walking_difficulty','handwriting_change','voice_change',
        'facial_expression_change','daily_activity_difficulty',
        'sleep_disorder','smell_loss','rigidity_issue'
    ]
    symptom_score = sum(answers[c] for c in q_cols)

    if confidence < 0.40 and symptom_score <= 10:
        risk = 'Low Risk'
        msg  = ('Your results suggest LOW risk of Parkinson disease. '
                'Continue routine check-ups and monitor any symptom changes.')
    elif confidence < 0.65 or symptom_score <= 22:
        risk = 'Medium Risk'
        msg  = ('Your results suggest MEDIUM risk. '
                'We recommend consulting a neurologist for clinical evaluation. '
                'Early detection significantly improves outcomes.')
    else:
        risk = 'High Risk'
        msg  = ('Your results suggest HIGH risk of Parkinson disease. '
                'Please consult a neurologist or movement disorder specialist promptly. '
                'This is a screening tool only -- clinical diagnosis is essential.')

    return {
        'ml_prediction' : int(pred),
        'ml_confidence' : round(float(confidence), 4),
        'symptom_score' : symptom_score,
        'risk_level'    : risk,
        'recommendation': msg
    }


# --- Test: Parkinson's-like input (using real dataset mean values) ---
pk_input = {
    'resting_tremor':2, 'bradykinesia':2, 'muscle_stiffness':2, 'balance_problem':2,
    'walking_difficulty':2, 'handwriting_change':2, 'voice_change':1,
    'facial_expression_change':1, 'daily_activity_difficulty':2,
    'sleep_disorder':2, 'smell_loss':2, 'rigidity_issue':2,
    'jitter':0.00699, 'shimmer':0.03366, 'fo':145.18, 'nhr':0.02921, 'hnr':20.97
}
r1 = predict_risk(pk_input)
print("=== Parkinson's-like input ===")
for k, v in r1.items(): print(f'  {k:18s}: {v}')

print()

# --- Test: Healthy-like input ---
hl_input = {
    'resting_tremor':0, 'bradykinesia':0, 'muscle_stiffness':0, 'balance_problem':0,
    'walking_difficulty':0, 'handwriting_change':0, 'voice_change':0,
    'facial_expression_change':0, 'daily_activity_difficulty':0,
    'sleep_disorder':0, 'smell_loss':0, 'rigidity_issue':0,
    'jitter':0.00387, 'shimmer':0.01762, 'fo':181.93, 'nhr':0.01148, 'hnr':24.67
}
r2 = predict_risk(hl_input)
print('=== Healthy-like input ===')
for k, v in r2.items(): print(f'  {k:18s}: {v}')

## Chatbot Integration Reference

### Load in chatbot backend (once at startup):
```python
import joblib, numpy as np
model  = joblib.load('parkinsons_model.pkl')
scaler = joblib.load('parkinsons_scaler.pkl')
cols   = joblib.load('feature_cols.pkl')
```

### Call after collecting all answers + voice features:
```python
result = predict_risk(chatbot_answers_dict)
print(result['risk_level'])       # Low / Medium / High Risk
print(result['ml_confidence'])    # 0.0 to 1.0
print(result['recommendation'])   # Display this to user
```

### Feature mapping (chatbot key -> real audio library output):
| Key | Audio Library Function |
|-----|------------------------|
| `fo` | fundamental frequency in Hz (e.g. librosa.yin) |
| `jitter` | cycle-to-cycle frequency variation (%) |
| `shimmer` | cycle-to-cycle amplitude variation |
| `nhr` | noise to harmonics ratio |
| `hnr` | harmonics to noise ratio (e.g. parselmouth) |